# Sales Prediction using Python

This project explores the relationship between advertising expenditure and sales using a 200-row advertising dataset. It compares Linear Regression and Random Forest Regression on a held-out test set. The results describe this dataset and split, not a guarantee of future business performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Libraries imported successfully!')

## 1. Load the Dataset

In [ ]:
# CodeAlpha-style advertising and sales dataset
url = 'https://raw.githubusercontent.com/selva86/datasets/master/Advertising.csv'
df = pd.read_csv(url)

# Remove the extra index column if it is present.
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
required_columns = ['TV', 'radio', 'newspaper', 'sales']
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f'Dataset is missing required columns: {missing_columns}')
df = df[required_columns].copy()
for column in required_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
if df[required_columns].isna().any().any():
    missing_counts = df[required_columns].isna().sum()
    raise ValueError(f'Dataset contains missing or non-numeric values:\n{missing_counts[missing_counts > 0]}')
if not np.isfinite(df[required_columns].to_numpy()).all():
    raise ValueError('Dataset contains infinite values.')
if (df['sales'] < 0).any():
    raise ValueError('Sales target contains negative values.')

print('Dataset loaded and validated successfully!')
print('Shape:', df.shape)
display(df.head())

## 2. Understand the Data

In [ ]:
print('Column names:')
print(df.columns.tolist())

print('\nData types:')
print(df.dtypes)

print('\nSummary statistics:')
display(df.describe())

## 3. Data Cleaning

In [ ]:
print('Missing values before cleaning:')
print(df.isnull().sum())

duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

print(f'\nDuplicate rows removed: {duplicate_count}')
print('Missing values after cleaning:')
print(df.isnull().sum())
print('\nShape after cleaning:', df.shape)

## 4. Exploratory Data Analysis

In [ ]:
# 4. Exploratory Data Analysis

# Distribution of Sales
plt.figure(figsize=(8, 5))
sns.histplot(df['sales'], kde=True)

plt.title('Distribution of Sales')
plt.xlabel('Sales')
plt.ylabel('Frequency')
plt.show()


# Advertising vs Sales
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# TV Advertising vs Sales
sns.scatterplot(data=df, x='TV', y='sales', ax=axes[0])
axes[0].set_title('TV Advertising vs Sales')
axes[0].set_xlabel('TV Advertising')
axes[0].set_ylabel('Sales')


# Radio Advertising vs Sales
sns.scatterplot(data=df, x='radio', y='sales', ax=axes[1])
axes[1].set_title('Radio Advertising vs Sales')
axes[1].set_xlabel('Radio Advertising')
axes[1].set_ylabel('Sales')


# Newspaper Advertising vs Sales
sns.scatterplot(data=df, x='newspaper', y='sales', ax=axes[2])
axes[2].set_title('Newspaper Advertising vs Sales')
axes[2].set_xlabel('Newspaper Advertising')
axes[2].set_ylabel('Sales')

plt.tight_layout()
plt.show()


# Correlation Heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    fmt='.2f',
    cmap='coolwarm'
)

plt.title('Correlation Heatmap')
plt.show()

## 5. Feature Selection

In [ ]:
# 5. Feature Selection

X = df[['TV', 'radio', 'newspaper']]
y = df['sales']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Features selected successfully!")
print("Features:", X.columns.tolist())
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 6. Train Linear Regression Model

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

print('Linear Regression model trained successfully!')

## 7. Train Random Forest Model

In [ ]:
random_forest_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)
random_forest_model.fit(X_train, y_train)
rf_pred = random_forest_model.predict(X_test)

print('Random Forest model trained successfully!')

## 8. Model Evaluation

In [ ]:
def evaluate_model(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    return [name, mae, rmse, r2]

results = pd.DataFrame([
    evaluate_model('Linear Regression', y_test, linear_pred),
    evaluate_model('Random Forest', y_test, rf_pred)
], columns=['Model', 'MAE', 'RMSE', 'R2 Score'])

display(results)

## 9. Actual vs Predicted Sales

In [ ]:
comparison = pd.DataFrame({
    'Actual Sales': y_test.values,
    'Predicted Sales': rf_pred
})

display(comparison.head(15))

plt.figure(figsize=(7, 5))
plt.scatter(y_test, rf_pred)
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Actual vs Predicted Sales - Random Forest')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle='--')
plt.show()

## 10. Advertising Feature Importance

In [ ]:
importance = pd.Series(
    random_forest_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

display(importance.to_frame('Importance'))

plt.figure(figsize=(7, 5))
importance.sort_values().plot(kind='barh')
plt.title('Advertising Feature Importance')
plt.xlabel('Importance')
plt.show()

## 11. Sales Prediction Example

In [ ]:
# Example prediction using new advertising spending values.
# Replace these values with your own planned advertising budget if needed.
new_advertising = pd.DataFrame({
    'TV': [150],
    'radio': [25],
    'newspaper': [20]
})

predicted_sales = random_forest_model.predict(new_advertising)[0]
print(f'Predicted sales for the example advertising budget: {predicted_sales:.2f}')

## 10. Key Insights

- TV and radio advertising show stronger relationships with sales than newspaper advertising in this dataset.
- Correlation describes association, not causation.
- The models are evaluated on a held-out 20% test split using a fixed random seed.
- Random Forest performs better than Linear Regression on this particular split. A single split is not enough to establish performance across all future data.
- Random Forest feature importance is model-specific and should not be interpreted as causal impact.
- Predictions are estimates based only on the three advertising variables. Other business factors are not represented.

## Conclusion

The notebook loads and validates the advertising dataset, explores its distributions and relationships, and compares two regression models using a held-out test set. On the saved run, Random Forest achieved MAE 0.63, RMSE 0.76 and R² 0.982, while Linear Regression achieved MAE 1.46, RMSE 1.78 and R² 0.899. These are results from one fixed split of this small dataset. They should not be treated as proof of production readiness or causal advertising impact.

## 12. Cross-validation and a simple baseline

The fixed 80/20 split above is useful for a reproducible demonstration, but it can be sensitive to which observations land in the test set. The following five-fold cross-validation compares both fitted models with a mean-prediction baseline. Each fold trains on four-fifths of the available observations and evaluates on the remaining fifth. These folds are internal validation, not external validation on a new market or time period.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=5, shuffle=True, random_state=42)
models_for_cv = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
}
scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2',
}

cv_rows = []
for model_name, estimator in models_for_cv.items():
    fold_scores = cross_validate(
        estimator, X, y, cv=cv, scoring=scoring, return_train_score=False,
        error_score='raise'
    )
    cv_rows.append({
        'Model': model_name,
        'MAE mean': -fold_scores['test_MAE'].mean(),
        'MAE std': fold_scores['test_MAE'].std(ddof=1),
        'RMSE mean': -fold_scores['test_RMSE'].mean(),
        'RMSE std': fold_scores['test_RMSE'].std(ddof=1),
        'R2 mean': fold_scores['test_R2'].mean(),
        'R2 std': fold_scores['test_R2'].std(ddof=1),
    })

cv_results = pd.DataFrame(cv_rows).set_index('Model')
display(cv_results.round(3))


### How to interpret the validation

- Lower MAE and RMSE indicate smaller prediction errors on the held-out folds.
- Higher R² indicates more explained variation relative to predicting the fold mean.
- The mean baseline is a minimum reference. A complex model should outperform it consistently before it is considered useful.
- The standard deviations show how much the scores vary across folds.
- This is still a small, single-source dataset. Do not interpret cross-validation as proof of causal impact or production readiness.